#Unsupervised Learning

## Cleaning up data

### Loading and Structuring Data

**Goal:** Load the CSV file and create a clean DataFrame with only 2 columns: `id` and `text`.

**Steps:**
1. Read the CSV file using pandas
2. Combine columns 2 and 3 into a single `text` column (using col3 if available, otherwise col2)
3. Keep only the `id` and `text` columns for further processing

In [ ]:
import pandas as pd

df = pd.read_csv("Student_Dataset.csv", sep=',', header=None, names=['id', 'col2', 'col3'])

df['text'] = df['col3'].fillna(df['col2'])
df = df[['id', 'text']]

print(df.head(10))
print(f"Shape: {df.shape}")

### Tokenization

**Goal:** Split each text into a list of individual words (tokens).

**Steps:**
1. Convert all text to lowercase using `.str.lower()` to ensure consistency (e.g., "Pain" and "pain" become the same word)
2. Split each text into words using `.str.split()` which separates by whitespace
3. Store the result in a new column called `words`

**Example:** `"The patient feels pain"` -> `["the", "patient", "feels", "pain"]`

In [ ]:
df['words'] = df['text'].str.lower().str.split()

print(df[['id', 'words']].head())

### Removing Stop Words

**Goal:** Remove common words and noise that don't carry meaningful information.

**Standard stop words:**
- Words like "the", "is", "at", "which", "on" appear frequently but don't help distinguish documents
- Scikit-learn provides a built-in list via `ENGLISH_STOP_WORDS`

**Custom stop words:**
- Contraction artifacts: `ve`, `re`, `ll`, `didn`, `doesn` are fragments created when TF-IDF splits contractions (`I've`, `we're`, `didn't`) on non-alphabetic characters
- Generic symptom verbs: words like `feel`, `feels`, `notice`, `just`, `like` appear across all clusters and don't help differentiate medical topics

**Punctuation stripping:**
- Simple whitespace splitting keeps trailing punctuation (e.g., `"eased,"`, `"garden,"`)
- Each token is cleaned with Python's built-in `re` module before filtering

**Example:** `["the", "patient,", "feels", "pain"]` → `["patient", "pain"]`

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(words):
    cleaned = []
    for word in words:
        word = re.sub(r'[^\w]', '', word)  # strip punctuation (commas, dots, etc.)
        if word and word not in stop_words and len(word) > 1:
            cleaned.append(word)
    return cleaned

df['words'] = df['words'].apply(clean_text)

print(f"Total stop words: {len(stop_words)}")
print("\nSample after cleaning:")
print(df[['id', 'words']].head())

## Vectorization (TF-IDF)

**Goal:** Convert text into numerical vectors that machine learning algorithms can process.

**What is TF-IDF?**
- **TF (Term Frequency):** How often a word appears in a document
  - Formula: `TF = count(word) / total_words_in_document`
- **IDF (Inverse Document Frequency):** How rare a word is across all documents
  - Formula: `IDF = log(total_documents / documents_containing_word)`
- **TF-IDF = TF × IDF**

**Key parameters used to reduce noise:**

| Parameter | Value |
|---|---|
| `max_df=0.6` | Ignore terms in >60% of documents |
| `min_df=3` | Ignore terms in <3 documents |
| `sublinear_tf=True` | Apply log(TF) instead of raw TF |
| `ngram_range=(1,2)` | Include single words and bigrams |
| `max_features=2000` | Keep top 2000 terms |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['clean_text'] = df['words'].apply(lambda x: ' '.join(x))

vectorizer = TfidfVectorizer(
    max_features=2000,
    max_df=0.6,
    min_df=3,
    sublinear_tf=True,
    ngram_range=(1, 2),
)
tfidf_matrix = vectorizer.fit_transform(df['clean_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

## Feature Enhancement

Before clustering, two additional transformations improve the quality of vector representations.

### Step 1 — Vector Normalization (Cosine Distance)

**Problem with K-Means and Euclidean distance:**
- K-Means minimizes Euclidean distances, but for text what matters is the *angle* between vectors, not their magnitude
- A long document and a short document on the same topic will have very different magnitudes but point in similar directions

**Solution:** Normalize each TF-IDF vector to unit length (L2 normalization):
- After normalization, all vectors have magnitude = 1
- Euclidean distance between unit vectors = `√(2 − 2 · cosine_similarity)`
- K-Means on normalized vectors is equivalent to **cosine K-Means**

**Result:** Documents are compared by topic similarity, regardless of length.

In [ ]:
from sklearn.preprocessing import normalize

tfidf_matrix = normalize(tfidf_matrix)

print("Vectors normalized to unit length (L2).")
print(f"Matrix shape: {tfidf_matrix.shape}")

### Step 2 — Dimensionality Reduction with LSA (TruncatedSVD)

**Problem with raw TF-IDF:**
- Vocabulary has 2000 dimensions → high dimensionality → "curse of dimensionality": distances become less meaningful
- `cough` and `coughing` are treated as completely unrelated terms even though they share the same meaning
- Noise and rare co-occurrences distort inter-document distances

**What is LSA (Latent Semantic Analysis)?**
- Applies **Truncated SVD** (Singular Value Decomposition) to the TF-IDF matrix
- Discovers hidden "topics" as linear combinations of words
- Example: `breathing`, `breath`, `shortness of breath` all load onto the same latent dimension
- Documents using *different words for the same concept* become closer in the reduced space

**Parameters:**

| Parameter | Value | Effect |
|---|---|---|
| `n_components=200` | Reduce to 200 latent dimensions | More components → more variance captured → better cluster separation |

**Result:** Documents with similar *meaning* are now closer, even if they use different words.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

lsa = TruncatedSVD(n_components=200, random_state=42)
tfidf_matrix = lsa.fit_transform(tfidf_matrix)
tfidf_matrix = normalize(tfidf_matrix)  # re-normalize after LSA: SVD changes vector magnitudes

explained = lsa.explained_variance_ratio_.sum()
print(f"LSA applied: {tfidf_matrix.shape[1]} latent dimensions")
print(f"Variance explained by 200 components: {explained:.2%}")
print("Vectors re-normalized to unit length after LSA.")

## Clustering with K-Means

**Goal:** Group similar documents together automatically without labels (unsupervised learning).

**What is K-Means?**
1. Choose K (number of clusters)
2. Randomly place K centroids (cluster centers)
3. Assign each document to the nearest centroid
4. Move centroids to the center of their assigned documents
5. Repeat steps 3-4 until centroids stop moving

**Key parameters:**
- `n_clusters`: Number of groups to create
- `random_state`: Ensures reproducibility (same results each run)
- `n_init`: Number of times to run with different starting centroids (keeps the best result)

### Finding Optimal K - Method 1: Elbow Method

**How it works:**
- Run K-Means for different values of K (2 to 10)
- Calculate the **inertia** for each K (sum of squared distances from each point to its centroid)
- Look for the "elbow" point where the decrease slows down

**Limitation:** Sometimes there is no clear elbow, making it hard to choose K.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(tfidf_matrix)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

### Finding Optimal K - Method 2: Silhouette Score

**Why use this method?**
- The Elbow Method above shows no clear elbow
- Silhouette Score gives a definitive answer

**How it works:**
- Score ranges from -1 to +1
- **+1** = points are well matched to their cluster
- **0** = points are on the border between clusters
- **-1** = points are in the wrong cluster

**Best K = highest Silhouette Score**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

silhouette_scores = []
K_range = range(2, 13)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(tfidf_matrix)
    score = silhouette_score(tfidf_matrix, labels, metric='cosine')
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.4f}")

best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f"\nBest K = {best_k} (highest score: {max(silhouette_scores):.4f})")

plt.figure(figsize=(10, 6))
plt.plot(K_range, silhouette_scores, 'go-')
plt.axvline(x=best_k, color='r', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score Method')
plt.legend()
plt.show()

### Applying K-Means

**Goal:** Assign each document to a cluster based on the chosen K value.

**What happens:**
- `fit_predict()` trains the model and returns cluster labels in one step
- Each document receives a label (0, 1, 2, ..., K-1)
- Documents with the same label are considered similar

**Output:** A new column `cluster` in the DataFrame showing which group each document belongs to.

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)

print(f"Using K = {best_k} clusters")
print(f"\nDistribution:")
print(df['cluster'].value_counts().sort_index())

## Visualization

### Method 1: PCA

**Limitation:** PCA preserves global variance but doesn't optimize for cluster separation. Clusters may overlap in 2D even if they are distinct in high dimensions.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(tfidf_matrix)  # already dense after LSA

plt.figure(figsize=(12, 8))

colors = plt.cm.tab20(range(best_k))
for i in range(best_k):
    mask = df['cluster'] == i
    plt.scatter(coords[mask, 0], coords[mask, 1],
                c=[colors[i]], label=f'Cluster {i}', alpha=0.7, s=50)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Cluster Visualization (PCA)')
plt.legend()
plt.show()

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

### Method 2: t-SNE

**Why t-SNE?**
- Designed specifically for visualization of high-dimensional data
- Preserves local neighborhoods (similar documents stay close)
- Much better at revealing cluster structure

**Note:** t-SNE is slower than PCA but produces clearer cluster separation.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
coords_tsne = tsne.fit_transform(tfidf_matrix)  # already dense after LSA

plt.figure(figsize=(12, 8))

colors = plt.cm.tab20(range(best_k))
for i in range(best_k):
    mask = df['cluster'] == i
    plt.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1],
                c=[colors[i]], label=f'Cluster {i}', alpha=0.7, s=50)

plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('Cluster Visualization (t-SNE)')
plt.legend()
plt.show()

## Cluster Analysis - Top Keywords

**Goal:** Understand what each cluster represents by looking at its most important words.

**How it works:**
- Each cluster has a **centroid** (center point) containing average TF-IDF values for all words
- Words with the highest values in the centroid are the most characteristic of that cluster
- We extract the top 10 words for each cluster

In [ ]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

# K-Means centroids are in LSA space — project back to original TF-IDF vocabulary space
# lsa.components_ has shape (n_components, n_features)
# cluster_centers_ has shape (n_clusters, n_components)
# dot product → (n_clusters, n_features): centroid coordinates in vocabulary space
centers_tfidf = kmeans.cluster_centers_ @ lsa.components_

for i in range(best_k):
    top_indices = centers_tfidf[i].argsort()[-10:][::-1]
    top_words = [feature_names[idx] for idx in top_indices]
    print(f"Cluster {i}: {', '.join(top_words)}")

## Results - Documents by Cluster

**Goal:** Display each document with its assigned cluster for easy review and validation.

In [ ]:
results_export = df[['id', 'text', 'cluster']].copy()
results_export = results_export.sort_values(['cluster', 'id'])

results_export.to_csv('results_clusters.csv', index=False)
print("Exported to 'results_clusters.csv'")

## Cluster Relevance Evaluation

**Goal:** Measure whether the obtained clusters are coherent and well-separated.

**Important caveat for text clustering:**
Geometric metrics like Silhouette often yield low scores even for semantically meaningful clusters. This is expected: after LSA projection, documents on overlapping topics (e.g. "chest pain" vs "shortness of breath") naturally sit close in vector space. A low global score does **not** mean the clustering is bad — always cross-check with the top keywords per cluster.

Three complementary metrics are used here:

| Metric | Range | Best value | Sensitive to |
|---|---|---|---|
| **Silhouette Score** | −1 → +1 | Close to +1 | Geometric separation |
| **Davies-Bouldin Index** | 0 → ∞ | Close to 0 | Compactness vs inter-cluster distance |
| **Calinski-Harabasz Index** | 0 → ∞ | Higher is better | Variance ratio between/within clusters |

### Silhouette Score
- For each document, measures how similar it is to its own cluster compared to other clusters
- Score ranges from **-1 to +1**:
  - **+1** → document is well matched to its cluster
  - **0** → document is on the border between two clusters
  - **-1** → document is likely in the wrong cluster
- Clusters with many negative or below-average scores indicate poor assignment

In [ ]:
from sklearn.metrics import calinski_harabasz_score, silhouette_score, silhouette_samples
import matplotlib.pyplot as plt
import numpy as np

labels = df['cluster']

# --- Global Silhouette Score ---
sil_global = silhouette_score(tfidf_matrix, labels, metric='cosine')
print(f"Global Silhouette Score: {sil_global:.4f}")
print("  → Close to 1 = well separated, close to 0 = overlapping boundaries, negative = misassigned.\n")

# --- Silhouette Score per cluster ---
sil_samples = silhouette_samples(tfidf_matrix, labels, metric='cosine')
df['silhouette'] = sil_samples

print("Average Silhouette Score per cluster:")
for i in sorted(df['cluster'].unique()):
    cluster_scores = df[df['cluster'] == i]['silhouette']
    print(f"  Cluster {i}: {cluster_scores.mean():.4f}  (n={len(cluster_scores)}, min={cluster_scores.min():.4f})")

# --- Visualization: silhouette per cluster ---
fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10
colors = plt.cm.tab20(range(best_k))

for i in sorted(df['cluster'].unique()):
    cluster_sil = np.sort(df[df['cluster'] == i]['silhouette'].values)
    size = len(cluster_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7, color=colors[i])
    ax.text(-0.05, y_lower + size / 2, str(i))
    y_lower = y_upper + 5

ax.axvline(x=sil_global, color='red', linestyle='--', label=f'Global score = {sil_global:.4f}')
ax.set_xlabel('Silhouette Score')
ax.set_ylabel('Cluster')
ax.set_title(f'Silhouette Score per Document (K={best_k})')
ax.legend()
plt.tight_layout()
plt.show()

### Davies-Bouldin Index

**What it measures:**
- For each cluster, computes the ratio: `(compactness of cluster A + compactness of cluster B) / distance(centroid A, centroid B)`
- Takes the **maximum** such ratio across all pairs — penalizes clusters that are loose internally *and* close to their neighbors
- Final score = average over all clusters

**Interpretation:**
- **Lower is better** (0 = perfect separation)
- Unlike Silhouette, it does not require pairwise document comparisons → faster on large datasets
- Less sensitive to outliers than Silhouette

**Important limitation for text/LSA data:**
- Davies-Bouldin uses **Euclidean distance** internally, applied to LSA-projected vectors
- In the LSA space, clusters of semantically related medical topics are naturally close and internally spread → scores above 3–4 are common and do **not** indicate a failed clustering
- This metric is most useful as a **relative comparison** across different K values, not as an absolute quality threshold

In [ ]:
from sklearn.metrics import davies_bouldin_score
import matplotlib.pyplot as plt
import numpy as np

db_score = davies_bouldin_score(tfidf_matrix, df['cluster'])
print(f"Davies-Bouldin Index: {db_score:.4f}")
print("  → Lower is better. For text/LSA data, scores of 3–6 are typical due to")
print("    natural overlap between topics. Use this metric to compare K values, not as absolute quality.\n")

# --- DB score across K values for comparison ---
from sklearn.cluster import KMeans

db_scores_k = []
K_range = range(2, 13)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(tfidf_matrix)
    db_scores_k.append(davies_bouldin_score(tfidf_matrix, labels_k))
    print(f"  K={k}: DB = {db_scores_k[-1]:.4f}")

best_k_db = K_range[db_scores_k.index(min(db_scores_k))]
print(f"\nBest K by Davies-Bouldin = {best_k_db} (score: {min(db_scores_k):.4f})")
print(f"Chosen K (Silhouette method) = {best_k}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, db_scores_k, 'ro-', linewidth=2)
ax.axvline(x=best_k_db, color='red', linestyle='--', alpha=0.7, label=f'Best K (DB) = {best_k_db}')
ax.axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'Chosen K (Silhouette) = {best_k}')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Davies-Bouldin Index')
ax.set_title('Davies-Bouldin Index by K (lower = better)')
ax.legend()
plt.tight_layout()
plt.show()

### Calinski-Harabasz Index

**What it measures:**
- Ratio of **between-cluster dispersion** (how far apart the cluster centroids are from the global centroid) to **within-cluster dispersion** (how spread documents are inside each cluster)
- Formula: `CH = (trace(B) / trace(W)) × ((n - k) / (k - 1))`
  - `B` = between-cluster scatter matrix, `W` = within-cluster scatter matrix
  - `n` = number of documents, `k` = number of clusters

**Interpretation:**
- **Higher is better** — no upper bound
- Very fast to compute (no pairwise distances)
- Works well for comparing different values of K on the same dataset

**Why it complements Silhouette:**
- Silhouette penalizes clusters with fuzzy borders even if their centroids are far apart
- Calinski-Harabasz rewards well-separated centroids regardless of individual point assignments
- For text data, this is often a more reliable signal than Silhouette

In [ ]:
from sklearn.metrics import calinski_harabasz_score

ch_score = calinski_harabasz_score(tfidf_matrix, df['cluster'])
print(f"Calinski-Harabasz Index: {ch_score:.2f}")
print("  → Higher is better. Compare across different K values to find the optimal number of clusters.\n")

# --- CH score across K values for comparison ---
from sklearn.cluster import KMeans

ch_scores = []
K_range = range(2, 13)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(tfidf_matrix)
    ch_scores.append(calinski_harabasz_score(tfidf_matrix, labels_k))
    print(f"  K={k}: CH = {ch_scores[-1]:.2f}")

best_k_ch = K_range[ch_scores.index(max(ch_scores))]
print(f"\nBest K by Calinski-Harabasz = {best_k_ch} (score: {max(ch_scores):.2f})")
print(f"Chosen K (Silhouette method) = {best_k}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, ch_scores, 'bs-', linewidth=2)
ax.axvline(x=best_k_ch, color='blue', linestyle='--', alpha=0.7, label=f'Best K (CH) = {best_k_ch}')
ax.axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'Chosen K (Silhouette) = {best_k}')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Calinski-Harabasz Score')
ax.set_title('Calinski-Harabasz Index by K')
ax.legend()
plt.tight_layout()
plt.show()